# 05 - Fair model bake-off

Head-to-head comparison of the four candidate models on the **matched estimand**:
`src.training.bakeoff_walk_forward` fits LogReg, XGBoost, HistGB (family-correct,
calibrated) **and** the hazard model on each decision-time fold's train, then scores
the **identical** test rows (same bookings, same label `cancel-by-arrival`, hazard at
d = min(lead, 14)). Every metric below is therefore apples-to-apples.

We judge with **several tools, not one**: ROC-AUC (ranking), PR-AUC (ranking at low
prevalence - the honest one here), Brier + reliability (calibration, because the
probabilities feed the cost decision), and confusion matrices (raw + normalised) at
each model's cost-optimal threshold. A **random-at-prevalence baseline** anchors
"chance".

## 0 - Setup

In [ ]:
from __future__ import annotations
import sys, time
from pathlib import Path
_t0=time.perf_counter()
def _step(m): print(f"  [{time.perf_counter()-_t0:5.2f}s] {m}", flush=True)
_here=Path.cwd().resolve()
while not (_here/"pyproject.toml").exists():
    if _here==_here.parent: raise RuntimeError("project root not found")
    _here=_here.parent
if str(_here) not in sys.path: sys.path.insert(0, str(_here))
import numpy as np, pandas as pd
import plotly.graph_objects as go, plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             roc_curve, precision_recall_curve, confusion_matrix)
from sklearn.calibration import calibration_curve
from src import color, figures_dir, tables_dir
import src.training as T, src.scoring as sc
pio.templates.default="plotly_white"
BRAND={n: color(n) for n in ["yellow","blue","green","orange","pink","purple","red"]}
MCOL={"logreg":BRAND["blue"],"xgboost":BRAND["orange"],"histgb":BRAND["green"],
      "hazard":BRAND["purple"],"baseline":"grey"}
FIG_DIR=figures_dir()/"05_comparison"; FIG_DIR.mkdir(parents=True, exist_ok=True)
TBL_DIR=tables_dir()/"05_comparison"; TBL_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE=42
_step("setup done (plotly).")

## 1 - Matched predictions + random baseline

`bakeoff_walk_forward` returns one row per test booking with every model's probability
on the SAME rows. The **random baseline** draws a hard cancel/not label ~ Bernoulli(p0)
at the test prevalence p0. NB: a *constant*-p0 predictor would give a degenerate
confusion matrix (all one class at any threshold); the **random draw** gives a
meaningful chance-level matrix - which is exactly why we use it here.

In [ ]:
_step("fair matched bake-off (fits 3 static + hazard per fold; heavy)...")
bake = T.bakeoff_walk_forward(n_folds=8, horizon_days=14, step_days=14, seed=RANDOM_STATE)
MODELS = ["logreg","xgboost","histgb","hazard"]
y = bake["y_true"].to_numpy()
P = {m: bake[f"p_{m}"].to_numpy() for m in MODELS}
p0 = float(y.mean())
rng = np.random.default_rng(RANDOM_STATE)
P["baseline"] = (rng.random(len(y)) < p0).astype(float)          # random Bernoulli(p0)
print(f"matched test rows: {len(y):,}  | test prevalence p0 = {p0:.3f} "
      f"(decision population; survivorship lowers it below the ~20% overall)")
print("  fairness check - all models scored on identical rows:", bake['y_true'].notna().all())

## 2 - Ranking: ROC-AUC and PR-AUC

ROC (rank quality overall) and Precision-Recall (rank quality where positives are
rare - the metric that actually matters at ~12% prevalence). No-skill references: the
ROC diagonal (AUC 0.5) and the PR line at p0.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("ROC curve", "Precision-Recall curve"))
for m in MODELS+["baseline"]:
    fpr,tpr,_ = roc_curve(y, P[m]); fig.add_scatter(x=fpr,y=tpr,mode="lines",name=m,legendgroup=m,
        line=dict(color=MCOL[m]), row=1,col=1)
    pr,rc,_ = precision_recall_curve(y, P[m]); fig.add_scatter(x=rc,y=pr,mode="lines",name=m,legendgroup=m,
        showlegend=False, line=dict(color=MCOL[m]), row=1,col=2)
fig.add_scatter(x=[0,1],y=[0,1],mode="lines",line=dict(color="black",dash="dot"),name="no-skill",row=1,col=1)
fig.add_scatter(x=[0,1],y=[p0,p0],mode="lines",line=dict(color="black",dash="dot"),showlegend=False,row=1,col=2)
fig.update_xaxes(title_text="FPR",row=1,col=1); fig.update_yaxes(title_text="TPR",row=1,col=1)
fig.update_xaxes(title_text="recall",row=1,col=2); fig.update_yaxes(title_text="precision",row=1,col=2)
fig.update_layout(title="Discrimination on the matched test set"); fig.show()

rank = pd.DataFrame({m: {"ROC_AUC": roc_auc_score(y,P[m]), "PR_AUC": average_precision_score(y,P[m]),
                        "Brier": brier_score_loss(y, np.clip(P[m],0,1))} for m in MODELS+["baseline"]}).T
display(rank.round(4))

## 3 - Calibration: reliability + Brier

The probabilities drive the overbooking math, so calibration is as important as
ranking. Reliability curves (predicted vs observed) + Brier (lower = better).
The random baseline's Brier equals the no-skill reference 2·p0·(1-p0).

In [ ]:
fig = go.Figure()
for m in MODELS:
    fp,mp = calibration_curve(y, np.clip(P[m],0,1), n_bins=10, strategy="quantile")
    fig.add_scatter(x=mp,y=fp,mode="lines+markers",name=m,line=dict(color=MCOL[m]))
mx=max(np.clip(P[m],0,1).max() for m in MODELS)
fig.add_scatter(x=[0,mx],y=[0,mx],mode="lines",line=dict(color="grey",dash="dash"),name="perfect")
fig.update_layout(title="Reliability (pooled matched OOS)", xaxis_title="mean predicted", yaxis_title="observed freq")
fig.show()

fig = go.Figure(go.Bar(x=list(rank.index), y=rank["Brier"], marker_color=[MCOL[m] for m in rank.index]))
fig.add_hline(y=2*p0*(1-p0), line=dict(color="black",dash="dot"), annotation_text="no-skill 2p0(1-p0)")
fig.update_layout(title="Brier score (lower = better)", yaxis_title="Brier"); fig.show()

## 4 - Confusion matrices (raw + normalised) at cost-optimal thresholds

Each model gets its own **cost-optimal** threshold (walk vs empty-room asymmetry);
the baseline uses its random labels. Raw counts in the table, row-normalised rates as
heatmaps - so a model that just flags everything (or nothing) is obvious.

In [ ]:
rows=[]; thr={}
for m in MODELS+["baseline"]:
    t = 0.5 if m=="baseline" else sc.cost_threshold_from_scores(y, P[m]); thr[m]=t
    pred = (P[m] >= t).astype(int)
    tn,fp,fn,tp = confusion_matrix(y, pred).ravel()
    prec = tp/(tp+fp) if tp+fp else 0.0; rec = tp/(tp+fn) if tp+fn else 0.0
    rows.append({"model":m,"threshold":round(t,3),"TP":tp,"FP":fp,"FN":fn,"TN":tn,
                 "precision":round(prec,3),"recall":round(rec,3),
                 "cost":sc.cost_at_threshold(y,P[m],t)["total_cost"]})
cm_tbl=pd.DataFrame(rows).set_index("model"); display(cm_tbl)

fig=make_subplots(rows=1, cols=len(MODELS)+1, subplot_titles=MODELS+["baseline"],
                  horizontal_spacing=0.04)
for i,m in enumerate(MODELS+["baseline"], start=1):
    pred=(P[m]>=thr[m]).astype(int); cm=confusion_matrix(y,pred)
    cmn=cm/cm.sum(axis=1, keepdims=True)
    txt=[[f"{cm[r,c]:,}<br>{cmn[r,c]:.0%}" for c in range(2)] for r in range(2)]
    fig.add_trace(go.Heatmap(z=cmn, x=["pred 0","pred 1"], y=["true 0","true 1"],
                  text=txt, texttemplate="%{text}", zmin=0, zmax=1, coloraxis="coloraxis"), row=1, col=i)
fig.update_layout(title="Confusion (colour = row-normalised rate; label = count + %)",
                  coloraxis=dict(colorscale="Blues")); fig.update_yaxes(autorange="reversed")
fig.show()

## 5 - Selection

Rank by **PR-AUC** (the right metric at ~12% prevalence), but only among models whose
**Brier** is within tolerance of the best (never ship a sharp-but-miscalibrated ranker,
since the probabilities feed the decision directly) - the rule in `src.scoring.best_model`.

In [ ]:
BRIER_TOL=0.005
cand={m:rank.loc[m] for m in MODELS}
best_brier=min(v["Brier"] for v in cand.values())
eligible=[m for m,v in cand.items() if v["Brier"]<=best_brier+BRIER_TOL]
winner=max(eligible, key=lambda m: cand[m]["PR_AUC"])
print("Brier-eligible:", eligible)
print(f"WINNER by PR-AUC among calibrated models: {winner}  "
      f"(PR-AUC={cand[winner]['PR_AUC']:.4f}, Brier={cand[winner]['Brier']:.4f})")
rank.to_csv(TBL_DIR/"model_ranking.csv"); cm_tbl.to_csv(TBL_DIR/"confusion.csv")

## 6 - What does the hazard buy us?

The static models give ONE score per booking; the hazard conditions on how far arrival
is. Its edge should concentrate **near arrival**. We (a) pair per-fold ΔAUC / ΔPR-AUC
(hazard - best static) and check signal > noise, and (b) split the test by remaining
horizon and compare AUC where the decision is most time-sensitive (d <= 3 days).

In [ ]:
best_static = max(MODELS[:-1], key=lambda m: rank.loc[m,"PR_AUC"])
per=[]
for k,g in bake.groupby("fold"):
    yy=g["y_true"].to_numpy()
    if len(np.unique(yy))<2: continue
    per.append({"fold":k,
                "dAUC": roc_auc_score(yy,g["p_hazard"]) - roc_auc_score(yy,g[f"p_{best_static}"]),
                "dPR":  average_precision_score(yy,g["p_hazard"]) - average_precision_score(yy,g[f"p_{best_static}"])})
per=pd.DataFrame(per)
print(f"hazard vs best static ({best_static}) - paired per fold:")
print(f"  mean dAUC = {per['dAUC'].mean():+.4f} +/- {per['dAUC'].std():.4f}  "
      f"(signal>noise: {per['dAUC'].mean()>per['dAUC'].std()})")
print(f"  mean dPR  = {per['dPR'].mean():+.4f} +/- {per['dPR'].std():.4f}")

# near-arrival edge: remaining horizon d<=3 (most time-sensitive) vs the rest
d = bake["d"].to_numpy()
for label, mask in [("near arrival (d<=3)", d<=3), ("rest (d>3)", d>3)]:
    yy=y[mask]
    if len(np.unique(yy))<2: continue
    ah=roc_auc_score(yy,P["hazard"][mask]); a_s=roc_auc_score(yy,P[best_static][mask])
    print(f"  {label:20s} n={mask.sum():>6,}: hazard AUC {ah:.3f} vs {best_static} {a_s:.3f}  (delta {ah-a_s:+.3f})")
fig=go.Figure()
fig.add_bar(x=per['fold'], y=per['dAUC'], marker_color=BRAND['purple'], name='dAUC (haz - static)')
fig.add_hline(y=0, line=dict(color='black'))
fig.update_layout(title=f'Per-fold hazard advantage over {best_static} (AUC)',
                  xaxis_title='fold', yaxis_title='delta AUC'); fig.show()
print('The time-resolved per-horizon proof (AUC rising toward arrival) lives in 08 section 7.')

## 7 - Explainability

Deep XAI is per-model: LogReg coefficients (01 §5), tree permutation importance + PDP
(02/03 §5), hazard feature importance + the days-until-arrival effect (08 §8). The
decisive differentiator is that only the hazard represents the *time* axis - visible as
its rising per-horizon AUC (08 §7) and the marginal-hazard-vs-days curve (08 §8).

## 8 - Verdict

All four models are compared on identical decision-time rows, judged by ranking
(ROC/PR-AUC), calibration (Brier/reliability) and cost-aware confusion matrices against
a random-at-prevalence baseline. Selection is PR-AUC among well-calibrated models. The
hazard model's value is the horizon awareness the static baselines cannot encode; where
that matters (near arrival) is where it separates - the rest is a genuine, honest tie
that the numbers above make explicit rather than assumed.